# Cluster Profiling and Classification Analysis using Random Forest

This notebook performs cluster profiling and classification analysis on global development indicators from the year 2022. The objective is to train a supervised machine learning model to predict the K-Means cluster labels (`Cluster_KMeans`) of countries based on their socioeconomic indicators.

This serves as a **Cluster Profiling and Validation** tool to understand the explicit boundaries and key socioeconomic drivers that distinguish different tiers of global development.

The analysis includes:
- Data loading and preparation
- Splitting data into training and test sets
- Training a Random Forest Classifier
- Evaluating the classifier using standard metrics (Accuracy, Classification Report)
- Analyzing Feature Importance to identify primary development drivers
- Validating the model using Stratified 5-Fold Cross-Validation
- Tuning hyperparameters using Grid Search (`GridSearchCV`)
- Extracting explicit decision rules using a simplified Decision Tree

## Importing libraries

In [ ]:
# 1. Importing the necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import classification_report, accuracy_score

In [ ]:
# Visual configuration for plots
sns.set_theme(style="whitegrid")

## Reading and loading the dataset

We load the dataset containing the country socioeconomic indicators along with the assigned K-Means cluster labels generated in the clustering notebook.

In [ ]:
# 2. Loading the data
# Using the output from the clustering notebook
df = pd.read_csv('../WDI-2022-clustering/WDI2022_clusters.csv')

# Rename the cluster column for convenience
df.rename(columns={'Cluster_KMeans': 'Cluster'}, inplace=True)

# The target variable is the 'Cluster' column
target_col = 'Cluster'

# Removing rows where the target might be empty (safety check)
df = df.dropna(subset=[target_col])

In [ ]:
df.columns

## Data preparation and feature selection

We drop non-predictive columns (`Country Name`, `Country Code`, `Region`) and the alternative cluster column (`Cluster_HDBSCAN`). We also drop `child_mortality_rate` to prevent direct demographic leakage, forcing the model to profile the development clusters using other socioeconomic and infrastructure indicators.

In [ ]:
# 3. Separating predictor variables (X) and target variable (y)
cols_to_drop = ['Country Name', 'Country Code', 'Region', 'child_mortality_rate', 
                'Cluster_HDBSCAN', target_col] 
X = df.drop(columns=[col for col in cols_to_drop if col in df.columns])
y = df[target_col]

## Train-Test Split

We split the data into 80% for training and 20% for testing. We use stratified splitting to ensure the cluster label proportions are preserved in both the training and test sets.

In [ ]:
# 4. Splitting into Train (80%) and Test (20%) datasets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

## Model Training

We train a Random Forest Classifier using 100 decision trees to classify countries into their respective development clusters.

In [ ]:
# 5. Training the Classification Model
print("Training the Random Forest Classifier model...")
rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42)
rf_classifier.fit(X_train, y_train)

## Model Predictions

We use the trained model to make predictions on the unseen test dataset.

In [ ]:
# 6. Making predictions on test data
y_pred = rf_classifier.predict(X_test)

## Model Evaluation

We evaluate the model's performance on the test set using R² (explained variance), Mean Absolute Error (MAE), and Root Mean Squared Error (RMSE).

In [ ]:
# 7. Evaluating the Model
accuracy = accuracy_score(y_test, y_pred)

print("\n--- Evaluation Metrics ---")
print(f"Accuracy: {accuracy:.4f} ({accuracy*100:.2f}% of countries correctly classified)")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

## Cluster Profiling and Data Leakage Analysis

Since the target variable (`Cluster`) was derived directly from the socioeconomic indicators in the clustering notebook, a very high accuracy is expected. The model is learning the deterministic mathematical boundaries drawn by the K-Means algorithm.

This task is essentially **Cluster Profiling**. Rather than being a pure predictive task for future unseen data (where there would be a conceptual data leakage), it serves to validate the clusters and identify which features are the most critical in separating countries into different tiers of development.

## Feature Importance

By extracting feature importances from the Random Forest, we can identify which socioeconomic indicators have the largest influence on defining global development tiers.

In [ ]:
# 9. Feature Importance Analysis
# We analyze which socioeconomic indicators are most important for defining the clusters
feature_importances = pd.Series(rf_classifier.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x=feature_importances.values, y=feature_importances.index, hue=feature_importances.index, palette="viridis", legend=False)
plt.title("Socioeconomic Indicators Importance in Cluster Profiling", fontsize=14)
plt.xlabel("Importance Score")
plt.ylabel("Socioeconomic Indicators")
plt.tight_layout()
plt.show()

## Stratified K-Fold Cross-Validation

Given the relatively small dataset, we apply 5-Fold Stratified Cross-Validation to obtain a robust and unbiased estimation of the classifier's performance across different partitions of the data.

In [ ]:
# 10. Stratified 5-Fold Cross-Validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(rf_classifier, X, y, cv=cv, scoring='accuracy')

print("--- 5-Fold Stratified Cross-Validation Results ---")
print(f"Accuracy scores for each fold: {cv_scores}")
print(f"Mean Accuracy: {cv_scores.mean() * 100:.2f}%")
print(f"Standard Deviation: {cv_scores.std() * 100:.2f}%")

## Hyperparameter Tuning

We use `GridSearchCV` to optimize the Random Forest hyperparameters and find the best combination of parameters like `n_estimators`, `max_depth`, and splitting criteria.

In [ ]:
# 11. Hyperparameter Tuning using GridSearchCV
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 5, 10],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=cv,
    scoring='accuracy',
    n_jobs=1
)

print("Running Hyperparameter Search...")
grid_search.fit(X_train, y_train)

print(f"Best parameters found: {grid_search.best_params_}")

# Evaluating the optimized model on the test set
best_model = grid_search.best_estimator_
best_pred = best_model.predict(X_test)

print(f"\nOptimized Model Accuracy on Test Set: {accuracy_score(y_test, best_pred)*100:.2f}%")
print("\nOptimized Model Classification Report:")
print(classification_report(y_test, best_pred))

## Explicit Decision Rules (Simplified Decision Tree)

To explain the classification logic visually, we train a single, simplified Decision Tree (restricted to depth 3) to map out the explicit socioeconomic rules that distinguish the country clusters.

In [ ]:
# 12. Training and Visualizing a Simplified Decision Tree
dt_classifier = DecisionTreeClassifier(max_depth=3, random_state=42)
dt_classifier.fit(X_train, y_train)

# Plot the decision tree
plt.figure(figsize=(16, 10))
plot_tree(
    dt_classifier,
    feature_names=X.columns,
    class_names=[str(c) for c in sorted(y.unique())],
    filled=True,
    rounded=True,
    fontsize=10
)
plt.title("Simplified Decision Tree for Cluster Profiling Rules", fontsize=16)
plt.tight_layout()
plt.show()

# Decision tree accuracy for reference
dt_pred = dt_classifier.predict(X_test)
print(f"Decision Tree Accuracy (max_depth=3): {accuracy_score(y_test, dt_pred)*100:.2f}%")